# Prepare the pain and pattern-cutting probes

This notebook demonstrates the complete probe-input workflow for the datasets in `legacy/pain_data` and `legacy/pattern_cutting_data`. It loads optode coordinates and channel pairings, normalizes the prepared mesh archive, registers each probe with `prepare_jacobian_probe`, and saves the prepared probe archive and diagnostic figure.

Run this notebook from the repository root. Registration uses the full tetrahedral mesh and can take several minutes per dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat

from mmc_nirs.light_transport.prepare_jacobian_probe import prepare_jacobian_probe
from mmc_nirs.light_transport.probe_utils import load_channel_pairs_from_snirf
from mmc_nirs.utils.prepared_input_io import resolve_prepared_input_path

REPOSITORY_ROOT = Path.cwd().resolve()
if not (REPOSITORY_ROOT / "mmc_nirs").is_dir():
    raise RuntimeError("Run this notebook from the mmc-nirs repository root")

PAIN_DIRECTORY = REPOSITORY_ROOT / "legacy" / "pain_data"
PATTERN_CUTTING_DIRECTORY = REPOSITORY_ROOT / "legacy" / "pattern_cutting_data"

## Shared mesh and figure helpers

Both legacy mesh archives use `nodes` and `elem` rather than the canonical `nodes` and `elements` keys expected by `prepare_jacobian_probe`. The fourth node column and fifth element column contain metadata, so only the first three coordinate columns and first four tetrahedral-index columns are passed to registration. One-based element indices are accepted and normalized internally.

In [ ]:
def load_prepared_mesh(path: Path) -> dict[str, np.ndarray]:
    """Load a legacy mesh archive in the format expected by probe preparation."""
    with np.load(path, allow_pickle=False) as archive:
        return {
            "nodes": archive["nodes"][:, :3].copy(),
            "elements": archive["elem"][:, :4].copy(),
        }


def save_registration_figure(path: Path) -> None:
    """Save and close the diagnostic figure created when plot=True."""
    figure = plt.gcf()
    figure.savefig(path, dpi=220, bbox_inches="tight", facecolor=figure.get_facecolor())
    plt.close(figure)

## Pain-data probe

The pain probe is stored as a MATLAB `SD` structure in `probe.SD`. `squeeze_me=True` and `struct_as_record=False` expose `SrcPos` and `DetPos` as normal attributes. Channel pairings come from `FingerTapping.snirf`.

`experiment_dir` controls where prepared inputs are written. `probefile` is resolved relative to that directory, so this example writes `notebook_probe.npz` inside `legacy/pain_data`.

In [ ]:
pain_config = {
    "filepaths": {
        "experiment_dir": PAIN_DIRECTORY,
        "meshfile": "mesh.npz",
        "probefile": "notebook_probe.npz",
    }
}

# Use the same path resolver used internally by prepare_jacobian_probe.
pain_mesh_path = resolve_prepared_input_path(pain_config, "meshfile")
pain_probe_output = resolve_prepared_input_path(pain_config, "probefile")

pain_mat = loadmat(PAIN_DIRECTORY / "probe.SD", squeeze_me=True, struct_as_record=False)
pain_sd = pain_mat["SD"]
pain_sources = np.asarray(pain_sd.SrcPos, dtype=float)
pain_detectors = np.asarray(pain_sd.DetPos, dtype=float)
pain_mesh = load_prepared_mesh(pain_mesh_path)
pain_pairings = load_channel_pairs_from_snirf(PAIN_DIRECTORY / "FingerTapping.snirf")

print(f"Pain sources: {len(pain_sources)}")
print(f"Pain detectors: {len(pain_detectors)}")
print(f"Pain channels: {len(pain_pairings)}")

Register the pain probe using millimetres, `LIA` input orientation, and a 20 mm short-separation threshold. `overwrite=True` ensures the notebook performs a fresh registration instead of loading an existing archive. `plot=True` creates the channel-aware registration diagnostic. The default embedding settings and `save_probe=True` behavior are retained.

In [ ]:
pain_probe = prepare_jacobian_probe(
    source_positions=pain_sources,
    detector_positions=pain_detectors,
    prepared_mesh=pain_mesh,
    units="mm",
    orientation="LIA",
    channel_pairings=pain_pairings,
    short_separation_flag="distance",
    short_separation_arg=20.0,
    experiment_config=pain_config,
    plot=True,
    overwrite=True,
)

pain_figure_output = PAIN_DIRECTORY / "notebook_registered_probe.png"
save_registration_figure(pain_figure_output)
print(f"Prepared pain probe: {pain_probe_output}")
print(f"Pain diagnostic: {pain_figure_output}")

## Pattern-cutting probe

The pattern-cutting `probe.mat` stores `sourcepos` and `detpos` directly as top-level arrays. Channel pairings come from `NIRS-2019-08-10_006.snirf`. Its prepared output and figure are written inside `legacy/pattern_cutting_data`.

In [ ]:
pattern_config = {
    "filepaths": {
        "experiment_dir": PATTERN_CUTTING_DIRECTORY,
        "meshfile": "mesh.npz",
        "probefile": "notebook_probe.npz",
    }
}

pattern_mesh_path = resolve_prepared_input_path(pattern_config, "meshfile")
pattern_probe_output = resolve_prepared_input_path(pattern_config, "probefile")

pattern_mat = loadmat(PATTERN_CUTTING_DIRECTORY / "probe.mat", squeeze_me=True, struct_as_record=False)
pattern_sources = np.asarray(pattern_mat["sourcepos"], dtype=float)
pattern_detectors = np.asarray(pattern_mat["detpos"], dtype=float)
pattern_mesh = load_prepared_mesh(pattern_mesh_path)
pattern_pairings = load_channel_pairs_from_snirf(
    PATTERN_CUTTING_DIRECTORY / "NIRS-2019-08-10_006.snirf"
)

print(f"Pattern-cutting sources: {len(pattern_sources)}")
print(f"Pattern-cutting detectors: {len(pattern_detectors)}")
print(f"Pattern-cutting channels: {len(pattern_pairings)}")

Register the pattern-cutting probe using millimetres, `RAS` input orientation, and a 14 mm short-separation threshold. As above, the prepared archive is saved automatically and the diagnostic figure is saved explicitly after registration.

In [ ]:
pattern_probe = prepare_jacobian_probe(
    source_positions=pattern_sources,
    detector_positions=pattern_detectors,
    prepared_mesh=pattern_mesh,
    units="mm",
    orientation="RAS",
    channel_pairings=pattern_pairings,
    short_separation_flag="distance",
    short_separation_arg=14.0,
    experiment_config=pattern_config,
    plot=True,
    overwrite=True,
)

pattern_figure_output = PATTERN_CUTTING_DIRECTORY / "notebook_registered_probe.png"
save_registration_figure(pattern_figure_output)
print(f"Prepared pattern-cutting probe: {pattern_probe_output}")
print(f"Pattern-cutting diagnostic: {pattern_figure_output}")

## Inspect the prepared outputs

Each returned dictionary and saved NPZ contains registered source/detector positions and directions, containing tetrahedron indices, the original channel pairings, and the derived short- and long-separation channel indices.

In [ ]:
for name, prepared in (("pain", pain_probe), ("pattern cutting", pattern_probe)):
    print(f"{name}:")
    print(f"  sources: {len(prepared['sourcepos'])}")
    print(f"  detectors: {len(prepared['detpos'])}")
    print(f"  channels: {len(prepared['channel_pairings'])}")
    print(f"  short separation: {len(prepared['short_separation_indices'])}")
    print(f"  long separation: {len(prepared['long_separation_indices'])}")